# SkyPoints – Data Analyst Assessment

## 1. Project Overview

The objective of this project is to analyze airline loyalty membership and redemption data to assess data quality, understand member distribution, evaluate membership activity and analyze redemption behaviour.

The project uses Python, PySpark, SQL and Databricks to transform raw source data into structured datasets suitable for analysis and reporting.

## 2. Business Objectives

- Analyze member profiles and membership attributes.
- Calculate member age and identify stale members.
- Identify missing, invalid and duplicate records.
- Understand member distribution across countries.
- Track changes in member information.
- Analyze redemption transactions, partners and transaction statuses.
- Calculate total redeemed miles.
- Validate consistency between member profiles and redemption records.
- Prepare reliable datasets for business reporting and visualization.

## 3. Technology Stack

- Databricks
- Python
- PySpark
- Spark SQL
- Delta Lake

## 4. Data Analysis Workflow

**Source Data → Data Cleaning → Transformation → Data Validation → Analysis → Reconciliation**

Member profile data and redemption transaction data are processed separately and connected using Member_ID for analytical purposes.

## 5. Key Business Assumptions

- Member_ID is used as the operational identifier for matching member profiles and redemption transactions. The source data dictionary identifies Member_Name as a key, so this discrepancy requires confirmation.
- DOB uses the MMddyyyy format.
- Enrollment_Date and Last_Flight_Date use the yyyyMMdd format.
- A member is classified as stale when the last flight date is more than 90 days before the processing date.
- Missing last-flight dates are classified as unknown.
- The latest available member record determines the current country.
- The supplied sample data is used to demonstrate the analytical workflow, not to claim production-scale performance.

## 6. Data Quality and Testing

The analysis includes mandatory-field checks, date validation, field-length checks, duplicate identification, redemption validation, unmatched member detection and country-wise reconciliation.

## 7. Expected Analytical Outputs

- Member distribution by country.
- Active and inactive member counts.
- Stale-member identification.
- Redemption analysis by partner and status.
- Total miles redeemed.
- Member-level redemption analysis.
- Data-quality and reconciliation results.

## 8. Future Improvements

Future enhancements include historical trend analysis, automated data ingestion, standardized country reference data, incremental processing, operational monitoring and Power BI dashboards.

In [0]:

# ============================================================
# SKYPOINTS - CELL 2: SOURCE DATA PREPARATION
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType
)
from pyspark.sql import functions as F
import json

# ------------------------------------------------------------
# 1. MEMBER FLAT FILE - ORIGINAL SOURCE FORMAT
# ------------------------------------------------------------

member_file_lines = [
    "|H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active",
    "|D|Elena|223457|20101012|20121013|GLD|Sam|CA|USA|03051985|A",
    "|D|Ravi|223458|20101012|20121013|SLV|Sam|TN|IND|03051985|A",
    "|D|Mateo|223459|20101012|20121013|GLD|Sam|NCR|PHIL|03051985|A",
    "|D|Nora|22345|20101012|20121013|PLT|Sam|ONT|CAN|03051985|A",
    "|D|Jacob|2256|20101012|20121013|SLV|Sam|VIC|AU|03051985|A"
]

# Preserve original source records
member_raw_schema = StructType([
    StructField("raw_record", StringType(), True)
])

members_raw_df = (
    spark.createDataFrame(
        [(line,) for line in member_file_lines],
        schema=member_raw_schema
    )
    .withColumn(
        "source_file_name",
        F.lit("skypoints_members_sample.txt")
    )
    .withColumn(
        "load_timestamp",
        F.current_timestamp()
    )
)

# ------------------------------------------------------------
# 2. SEPARATE HEADER AND DETAIL RECORDS
# ------------------------------------------------------------

header_df = members_raw_df.filter(
    F.col("raw_record").startswith("|H|")
)

member_detail_raw_df = members_raw_df.filter(
    F.col("raw_record").startswith("|D|")
)

# ------------------------------------------------------------
# 3. DEFINE EXPECTED SOURCE COLUMNS
# ------------------------------------------------------------

member_columns = [
    "Member_Name",
    "Member_Id",
    "Enrollment_Date",
    "Last_Flight_Date",
    "Tier_Code",
    "Agent_Name",
    "State",
    "Country",
    "DOB",
    "Is_Active"
]

# ------------------------------------------------------------
# 4. DISTRIBUTED PYSPARK PARSING
# ------------------------------------------------------------

# Split records using Spark transformations.
# No collect() or Python row-by-row loop is used.

member_split_df = member_detail_raw_df.withColumn(
    "_fields",
    F.split(F.col("raw_record"), r"\|", -1)
)

# A valid record must contain 12 split elements:
# empty value, D indicator and 10 member fields.

valid_format_df = member_split_df.filter(
    F.size(F.col("_fields")) == 12
)

malformed_member_df = member_split_df.filter(
    F.size(F.col("_fields")) != 12
)

# Extract the 10 member fields using Spark array functions.

members_source_df = valid_format_df.select(
    *[
        F.element_at(F.col("_fields"), position).alias(column_name)
        for position, column_name in enumerate(member_columns, start=3)
    ],
    "source_file_name",
    "load_timestamp"
)

# ------------------------------------------------------------
# 5. PRESERVE MALFORMED RECORDS FOR REVIEW
# ------------------------------------------------------------

malformed_member_df = malformed_member_df.select(
    "raw_record",
    "source_file_name",
    "load_timestamp"
)

malformed_member_df.createOrReplaceTempView(
    "malformed_member_records"
)

# ------------------------------------------------------------
# 6. JSON REDEMPTION SOURCE
# ------------------------------------------------------------

redemption_json = {
    "member_id": "223457",
    "feed_date": "20240115",
    "redemptions": [
        {
            "txn_id": "RX10091",
            "txn_date": "20240110",
            "partner": "AeroLink",
            "miles_redeemed": 12000,
            "status": "COMPLETED"
        },
        {
            "txn_id": "RX10092",
            "txn_date": "20240113",
            "partner": "SkyPoints",
            "miles_redeemed": 5000,
            "status": "PENDING"
        }
    ]
}

redemption_raw_schema = StructType([
    StructField("raw_json", StringType(), True)
])

redemption_raw_df = (
    spark.createDataFrame(
        [(json.dumps(redemption_json),)],
        schema=redemption_raw_schema
    )
    .withColumn(
        "source_file_name",
        F.lit("skypoints_redemptions_sample.json")
    )
    .withColumn(
        "load_timestamp",
        F.current_timestamp()
    )
)

redemption_raw_df.createOrReplaceTempView(
    "redemption_source"
)

# ------------------------------------------------------------
# 7. SOURCE VALIDATION SUMMARY
# ------------------------------------------------------------

source_count = members_raw_df.count()
header_count = header_df.count()
detail_count = member_detail_raw_df.count()
parsed_count = members_source_df.count()
malformed_count = malformed_member_df.count()

print("MEMBER SOURCE VALIDATION")
print("Total raw file records:", source_count)
print("Header records:", header_count)
print("Detail records:", detail_count)
print("Successfully parsed records:", parsed_count)
print("Malformed records:", malformed_count)

print("\nREDEMPTION SOURCE VALIDATION")
print("Expected JSON documents: 1")
print("Loaded JSON documents:", redemption_raw_df.count())

print("\nPARSED MEMBER DATA")
display(members_source_df)

print("\nMALFORMED MEMBER RECORDS")
display(malformed_member_df)

print("\nRAW MEMBER FILE")
display(members_raw_df)

print("\nRAW REDEMPTION JSON")
display(redemption_raw_df)


MEMBER SOURCE VALIDATION
Total raw file records: 6
Header records: 1
Detail records: 5
Successfully parsed records: 5
Malformed records: 0

REDEMPTION SOURCE VALIDATION
Expected JSON documents: 1
Loaded JSON documents: 1

PARSED MEMBER DATA


Member_Name,Member_Id,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,DOB,Is_Active,source_file_name,load_timestamp
Elena,223457,20101012,20121013,GLD,Sam,CA,USA,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:09.630Z
Ravi,223458,20101012,20121013,SLV,Sam,TN,IND,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:09.630Z
Mateo,223459,20101012,20121013,GLD,Sam,NCR,PHIL,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:09.630Z
Nora,22345,20101012,20121013,PLT,Sam,ONT,CAN,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:09.630Z
Jacob,2256,20101012,20121013,SLV,Sam,VIC,AU,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:09.630Z



MALFORMED MEMBER RECORDS


raw_record,source_file_name,load_timestamp



RAW MEMBER FILE


raw_record,source_file_name,load_timestamp
|H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z
|D|Elena|223457|20101012|20121013|GLD|Sam|CA|USA|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z
|D|Ravi|223458|20101012|20121013|SLV|Sam|TN|IND|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z
|D|Mateo|223459|20101012|20121013|GLD|Sam|NCR|PHIL|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z
|D|Nora|22345|20101012|20121013|PLT|Sam|ONT|CAN|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z
|D|Jacob|2256|20101012|20121013|SLV|Sam|VIC|AU|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:10.690Z



RAW REDEMPTION JSON


raw_json,source_file_name,load_timestamp
"{""member_id"": ""223457"", ""feed_date"": ""20240115"", ""redemptions"": [{""txn_id"": ""RX10091"", ""txn_date"": ""20240110"", ""partner"": ""AeroLink"", ""miles_redeemed"": 12000, ""status"": ""COMPLETED""}, {""txn_id"": ""RX10092"", ""txn_date"": ""20240113"", ""partner"": ""SkyPoints"", ""miles_redeemed"": 5000, ""status"": ""PENDING""}]}",skypoints_redemptions_sample.json,2026-11-02T13:36:11.221Z


In [0]:

%sql

CREATE DATABASE IF NOT EXISTS skypoints_db;

USE skypoints_db;

-- Raw landing table for original member file records
CREATE TABLE IF NOT EXISTS raw_member_file_records (
    raw_record STRING,
    source_file_name STRING,
    load_timestamp TIMESTAMP
)
USING DELTA;

-- Raw landing table for original redemption JSON documents
CREATE TABLE IF NOT EXISTS raw_redemption_json_documents (
    raw_json STRING,
    source_file_name STRING,
    load_timestamp TIMESTAMP
)
USING DELTA;


In [0]:
# Persist raw member file records
members_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("skypoints_db.raw_member_file_records")

# Persist raw redemption JSON
redemption_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("skypoints_db.raw_redemption_json_documents")

# Verify loaded records
print("Raw member records:",
      spark.table("skypoints_db.raw_member_file_records").count())

print("Raw redemption documents:",
      spark.table("skypoints_db.raw_redemption_json_documents").count())

display(spark.table("skypoints_db.raw_member_file_records"))
display(spark.table("skypoints_db.raw_redemption_json_documents"))

Raw member records: 6
Raw redemption documents: 1


raw_record,source_file_name,load_timestamp
|H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z
|D|Elena|223457|20101012|20121013|GLD|Sam|CA|USA|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z
|D|Ravi|223458|20101012|20121013|SLV|Sam|TN|IND|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z
|D|Mateo|223459|20101012|20121013|GLD|Sam|NCR|PHIL|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z
|D|Nora|22345|20101012|20121013|PLT|Sam|ONT|CAN|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z
|D|Jacob|2256|20101012|20121013|SLV|Sam|VIC|AU|03051985|A,skypoints_members_sample.txt,2026-11-02T13:36:22.466Z


raw_json,source_file_name,load_timestamp
"{""member_id"": ""223457"", ""feed_date"": ""20240115"", ""redemptions"": [{""txn_id"": ""RX10091"", ""txn_date"": ""20240110"", ""partner"": ""AeroLink"", ""miles_redeemed"": 12000, ""status"": ""COMPLETED""}, {""txn_id"": ""RX10092"", ""txn_date"": ""20240113"", ""partner"": ""SkyPoints"", ""miles_redeemed"": 5000, ""status"": ""PENDING""}]}",skypoints_redemptions_sample.json,2026-11-02T13:36:29.832Z


In [0]:
%sql

USE skypoints_db;

CREATE TABLE IF NOT EXISTS stg_members (
    member_name STRING,
    member_id STRING,
    enrollment_date DATE,
    last_flight_date DATE,
    tier_code STRING,
    agent_name STRING,
    state STRING,
    country STRING,
    dob DATE,
    is_active STRING,
    age INT,
    stale_member BOOLEAN,
    source_file_name STRING,
    load_timestamp TIMESTAMP,
    processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
print("MEMBER SOURCE SCHEMA")
members_source_df.printSchema()

print("MEMBER SOURCE SAMPLE")
display(members_source_df.limit(5))

print("RAW MEMBER LANDING TABLE")
spark.table("skypoints_db.raw_member_file_records").printSchema()

print("STAGING TABLE")
spark.table("skypoints_db.stg_members").printSchema()

MEMBER SOURCE SCHEMA
root
 |-- Member_Name: string (nullable = true)
 |-- Member_Id: string (nullable = true)
 |-- Enrollment_Date: string (nullable = true)
 |-- Last_Flight_Date: string (nullable = true)
 |-- Tier_Code: string (nullable = true)
 |-- Agent_Name: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- load_timestamp: timestamp (nullable = false)

MEMBER SOURCE SAMPLE


Member_Name,Member_Id,Enrollment_Date,Last_Flight_Date,Tier_Code,Agent_Name,State,Country,DOB,Is_Active,source_file_name,load_timestamp
Elena,223457,20101012,20121013,GLD,Sam,CA,USA,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:37.315Z
Ravi,223458,20101012,20121013,SLV,Sam,TN,IND,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:37.315Z
Mateo,223459,20101012,20121013,GLD,Sam,NCR,PHIL,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:37.315Z
Nora,22345,20101012,20121013,PLT,Sam,ONT,CAN,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:37.315Z
Jacob,2256,20101012,20121013,SLV,Sam,VIC,AU,03051985,A,skypoints_members_sample.txt,2026-11-02T13:36:37.315Z


RAW MEMBER LANDING TABLE
root
 |-- raw_record: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)

STAGING TABLE
root
 |-- member_name: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- enrollment_date: date (nullable = true)
 |-- last_flight_date: date (nullable = true)
 |-- tier_code: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- is_active: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- stale_member: boolean (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read source data
source_df = members_source_df

# 2. Safely parse dates
transformed_df = (
    source_df
    .withColumn(
        "enrollment_date",
        F.to_date(F.expr(
            "try_to_timestamp(Enrollment_Date, 'yyyyMMdd')"
        ))
    )
    .withColumn(
        "last_flight_date",
        F.to_date(F.expr(
            "try_to_timestamp(Last_Flight_Date, 'yyyyMMdd')"
        ))
    )
    .withColumn(
        "dob",
        F.to_date(F.expr(
            "try_to_timestamp(DOB, 'MMddyyyy')"
        ))
    )
    .withColumn("normalized_member_id", F.trim("Member_Id"))
)

# 3. Identify duplicate Member IDs using a window function
duplicate_window = Window.partitionBy("normalized_member_id")

transformed_df = transformed_df.withColumn(
    "duplicate_count",
    F.when(
        F.col("normalized_member_id").isNotNull() &
        (F.col("normalized_member_id") != ""),
        F.count("*").over(duplicate_window)
    ).otherwise(0)
)

# 4. Data quality checks
rejection_checks = [
    F.when(
        F.col("Member_Name").isNull() |
        (F.trim("Member_Name") == ""),
        "MISSING_MEMBER_NAME"
    ),
    F.when(
        F.col("Member_Id").isNull() |
        (F.trim("Member_Id") == ""),
        "MISSING_MEMBER_ID"
    ),
    F.when(
        F.col("Enrollment_Date").isNull() |
        (F.trim("Enrollment_Date") == ""),
        "MISSING_ENROLLMENT_DATE"
    ),
    F.when(
        (F.trim("Enrollment_Date") != "") &
        F.col("enrollment_date").isNull(),
        "INVALID_ENROLLMENT_DATE"
    ),
    F.when(
        F.col("enrollment_date") > F.current_date(),
        "FUTURE_ENROLLMENT_DATE"
    ),
    F.when(
        (F.trim("Last_Flight_Date") != "") &
        F.col("last_flight_date").isNull(),
        "INVALID_LAST_FLIGHT_DATE"
    ),
    F.when(
        F.col("last_flight_date") > F.current_date(),
        "FUTURE_LAST_FLIGHT_DATE"
    ),
    F.when(
        (F.trim("DOB") != "") &
        (
            F.col("dob").isNull() |
            (F.col("dob") > F.current_date())
        ),
        "INVALID_DOB"
    ),
    F.when(F.length(F.trim("Member_Name")) > 255, "MEMBER_NAME_TOO_LONG"),
    F.when(F.length(F.trim("Member_Id")) > 18, "MEMBER_ID_TOO_LONG"),
    F.when(F.length(F.trim("Tier_Code")) > 5, "TIER_CODE_TOO_LONG"),
    F.when(F.length(F.trim("Agent_Name")) > 255, "AGENT_NAME_TOO_LONG"),
    F.when(F.length(F.trim("State")) > 5, "STATE_TOO_LONG"),
    F.when(F.length(F.trim("Country")) > 5, "COUNTRY_TOO_LONG"),
    F.when(
        F.col("Is_Active").isNotNull() &
        (~F.upper(F.trim("Is_Active")).isin("A", "I")),
        "INVALID_ACTIVE_FLAG"
    ),
    F.when(
        F.col("duplicate_count") > 1,
        "DUPLICATE_MEMBER_ID"
    )
]

transformed_df = transformed_df.withColumn(
    "rejection_reason",
    F.concat_ws(";", *rejection_checks)
)

# 5. Separate valid and rejected records
valid_df = transformed_df.filter(F.col("rejection_reason") == "")
rejected_df = transformed_df.filter(F.col("rejection_reason") != "")

# 6. Calculate age and stale-member status
staging_df = (
    valid_df
    .withColumn(
        "age",
        F.when(
            F.col("dob").isNotNull(),
            F.floor(
                F.months_between(F.current_date(), F.col("dob")) / 12
            ).cast("int")
        )
    )
    .withColumn(
        "stale_member",
        F.when(
            F.col("last_flight_date").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.datediff(F.current_date(), F.col("last_flight_date")) > 90
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .select(
        F.col("Member_Name").alias("member_name"),
        F.col("Member_Id").alias("member_id"),
        "enrollment_date",
        "last_flight_date",
        F.col("Tier_Code").alias("tier_code"),
        F.col("Agent_Name").alias("agent_name"),
        F.col("State").alias("state"),
        F.col("Country").alias("country"),
        "dob",
        F.col("Is_Active").alias("is_active"),
        "age",
        "stale_member",
        "source_file_name",
        "load_timestamp",
        "processed_timestamp"
    )
)

# 7. Load valid records into staging
staging_df.createOrReplaceTempView("stg_members_ready")

spark.sql("""
    INSERT OVERWRITE TABLE skypoints_db.stg_members
    SELECT * FROM stg_members_ready
""")

# 8. Save rejected records
(
    rejected_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skypoints_db.quarantine_members")
)

# 9. Save duplicate records for investigation
(
    transformed_df
    .filter(F.col("duplicate_count") > 1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skypoints_db.duplicate_member_review")
)

# 10. Display results
print("SOURCE RECORDS:", source_df.count())
print("VALID STAGING RECORDS:", staging_df.count())
print("REJECTED RECORDS:", rejected_df.count())
print(
    "DUPLICATE MEMBER RECORDS:",
    transformed_df.filter(F.col("duplicate_count") > 1).count()
)

print("STAGING TABLE")
display(spark.table("skypoints_db.stg_members"))

print("QUARANTINED RECORDS")
display(spark.table("skypoints_db.quarantine_members"))

print("DUPLICATE MEMBER REVIEW")
display(spark.table("skypoints_db.duplicate_member_review"))


SOURCE RECORDS: 5
VALID STAGING RECORDS: 5
REJECTED RECORDS: 0
DUPLICATE MEMBER RECORDS: 0
STAGING TABLE


member_name,member_id,enrollment_date,last_flight_date,tier_code,agent_name,state,country,dob,is_active,age,stale_member,source_file_name,load_timestamp,processed_timestamp
Nora,22345,2010-10-12,2012-10-13,PLT,Sam,ONT,CAN,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Elena,223457,2010-10-12,2012-10-13,GLD,Sam,CA,USA,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Ravi,223458,2010-10-12,2012-10-13,SLV,Sam,TN,IND,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Mateo,223459,2010-10-12,2012-10-13,GLD,Sam,NCR,PHIL,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Jacob,2256,2010-10-12,2012-10-13,SLV,Sam,VIC,AU,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z


QUARANTINED RECORDS


Member_Name,Member_Id,enrollment_date,last_flight_date,Tier_Code,Agent_Name,State,Country,dob,Is_Active,source_file_name,load_timestamp,normalized_member_id,duplicate_count,rejection_reason


DUPLICATE MEMBER REVIEW


Member_Name,Member_Id,enrollment_date,last_flight_date,Tier_Code,Agent_Name,State,Country,dob,Is_Active,source_file_name,load_timestamp,normalized_member_id,duplicate_count,rejection_reason


In [0]:

%sql

USE skypoints_db;

-- Create country-specific target tables
-- Each table uses the same schema as the staging table.

CREATE TABLE IF NOT EXISTS target_usa
LIKE stg_members;

CREATE TABLE IF NOT EXISTS target_ind
LIKE stg_members;

CREATE TABLE IF NOT EXISTS target_phil
LIKE stg_members;

CREATE TABLE IF NOT EXISTS target_can
LIKE stg_members;

CREATE TABLE IF NOT EXISTS target_au
LIKE stg_members;

-- Verify that all target tables exist
SHOW TABLES;


database,tableName,isTemporary
skypoints_db,duplicate_member_review,false
skypoints_db,final_member_redemptions,false
skypoints_db,flattened_redemptions,false
skypoints_db,member_current,false
skypoints_db,members_au,false
skypoints_db,members_can,false
skypoints_db,members_incremental_demo,false
skypoints_db,members_ind,false
skypoints_db,members_phil,false
skypoints_db,members_usa,false


In [0]:

from pyspark.sql import functions as F

# Read validated staging data
staging_df = spark.table("skypoints_db.stg_members")

# Country-to-table mapping
country_tables = {
    "USA": "target_usa",
    "IND": "target_ind",
    "PHIL": "target_phil",
    "CAN": "target_can",
    "AU": "target_au"
}

# Load records into country-specific tables
for country, table_name in country_tables.items():

    country_df = staging_df.filter(
        F.upper(F.trim(F.col("country"))) == country
    )

    country_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"skypoints_db.{table_name}")

    record_count = spark.table(
        f"skypoints_db.{table_name}"
    ).count()

    print(f"{country}: {record_count} records")

# Verify total records
total_country_records = sum(
    spark.table(f"skypoints_db.{table_name}").count()
    for table_name in country_tables.values()
)

staging_count = staging_df.count()

print("--------------------------------")
print("Staging records:", staging_count)
print("Country table records:", total_country_records)
print("Reconciliation:",
      "PASSED" if staging_count == total_country_records else "FAILED")


USA: 1 records
IND: 1 records
PHIL: 1 records
CAN: 1 records
AU: 1 records
--------------------------------
Staging records: 5
Country table records: 5
Reconciliation: PASSED


In [0]:

%sql

USE skypoints_db;

-- Create central member master table
CREATE TABLE IF NOT EXISTS member_current
LIKE stg_members;

-- Verify table creation
DESCRIBE member_current;


col_name,data_type,comment
member_name,string,null
member_id,string,null
enrollment_date,date,null
last_flight_date,date,null
tier_code,string,null
agent_name,string,null
state,string,null
country,string,null
dob,date,null
is_active,string,null


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read validated staging records
staging_df = spark.table("skypoints_db.stg_members")

# Member_ID is used as the operational identifier.
# The data dictionary marks Member_Name as a key.
# Confirm this discrepancy with the source owner.

# Check duplicate member IDs
duplicate_ids = (
    staging_df
    .groupBy("member_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate member IDs:", duplicate_ids.count())
display(duplicate_ids)

# Identify records with missing load timestamps
missing_timestamp = staging_df.filter(
    F.col("load_timestamp").isNull()
)

print("Records with missing load timestamp:",
      missing_timestamp.count())

# Identify ambiguous records with identical timestamps
ambiguous_ids = (
    staging_df
    .groupBy("member_id", "load_timestamp", "processed_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .select("member_id")
    .distinct()
)

# Exclude records that cannot be reliably ordered
safe_source = (
    staging_df
    .filter(F.col("load_timestamp").isNotNull())
    .join(ambiguous_ids, on="member_id", how="left_anti")
)

# Select latest available record per member
window_spec = (
    Window
    .partitionBy("member_id")
    .orderBy(
        F.col("load_timestamp").desc(),
        F.col("processed_timestamp").desc_nulls_last()
    )
)

latest_records = (
    safe_source
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_records.createOrReplaceTempView("latest_member_records")

# Merge latest records into the master table
spark.sql("""
    MERGE INTO skypoints_db.member_current AS target
    USING latest_member_records AS source
    ON target.member_id = source.member_id

    WHEN MATCHED AND (
        source.load_timestamp > target.load_timestamp
    )
    THEN UPDATE SET *

    WHEN NOT MATCHED
    THEN INSERT *
""")

print("Current member master records:",
      spark.table("skypoints_db.member_current").count())

display(spark.table("skypoints_db.member_current"))


Duplicate member IDs: 0


member_id,count


Records with missing load timestamp: 0
Current member master records: 5


member_name,member_id,enrollment_date,last_flight_date,tier_code,agent_name,state,country,dob,is_active,age,stale_member,source_file_name,load_timestamp,processed_timestamp
Elena,223457,2010-10-12,2012-10-13,GLD,Sam,KA,IND,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-07T10:05:42.206Z,2026-11-02T11:45:31.819Z
Nora,22345,2010-10-12,2012-10-13,PLT,Sam,ONT,CAN,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Ravi,223458,2010-10-12,2012-10-13,SLV,Sam,TN,IND,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Mateo,223459,2010-10-12,2012-10-13,GLD,Sam,NCR,PHIL,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z
Jacob,2256,2010-10-12,2012-10-13,SLV,Sam,VIC,AU,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-02T13:36:40.488Z,2026-11-02T13:36:40.488Z


In [0]:

from pyspark.sql import functions as F

# Read the current member master
current_df = spark.table("skypoints_db.member_current")

country_tables = {
    "USA": "target_usa",
    "IND": "target_ind",
    "PHIL": "target_phil",
    "CAN": "target_can",
    "AU": "target_au"
}

# Synchronize country tables from the current master
for country, table_name in country_tables.items():

    country_df = current_df.filter(
        F.upper(F.trim(F.col("country"))) == country
    )

    country_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"skypoints_db.{table_name}")

    record_count = spark.table(
        f"skypoints_db.{table_name}"
    ).count()

    print(f"{country}: {record_count} records")

# Final reconciliation
total_country_records = sum(
    spark.table(f"skypoints_db.{table_name}").count()
    for table_name in country_tables.values()
)

master_count = current_df.count()

print("--------------------------------")
print("Current master records:", master_count)
print("Country table records:", total_country_records)

print(
    "Reconciliation:",
    "PASSED" if master_count == total_country_records else "FAILED"
)


USA: 0 records
IND: 2 records
PHIL: 1 records
CAN: 1 records
AU: 1 records
--------------------------------
Current master records: 5
Country table records: 5
Reconciliation: PASSED


In [0]:

%sql

USE skypoints_db;

-- Create the redemption transaction table
CREATE TABLE IF NOT EXISTS redemption_transactions (
    member_id STRING,
    feed_date DATE,
    txn_id STRING,
    txn_date DATE,
    partner STRING,
    miles_redeemed BIGINT,
    status STRING,
    source_file_name STRING,
    load_timestamp TIMESTAMP,
    processed_timestamp TIMESTAMP
)
USING DELTA;

-- Verify table structure
DESCRIBE redemption_transactions;


col_name,data_type,comment
member_id,string,null
feed_date,date,null
txn_id,string,null
txn_date,date,null
partner,string,null
miles_redeemed,bigint,null
status,string,null
source_file_name,string,null
load_timestamp,timestamp,null
processed_timestamp,timestamp,null


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    ArrayType, LongType
)

# 1. Define JSON schema
redemption_schema = StructType([
    StructField("member_id", StringType(), True),
    StructField("feed_date", StringType(), True),
    StructField(
        "redemptions",
        ArrayType(
            StructType([
                StructField("txn_id", StringType(), True),
                StructField("txn_date", StringType(), True),
                StructField("partner", StringType(), True),
                StructField("miles_redeemed", LongType(), True),
                StructField("status", StringType(), True)
            ])
        ),
        True
    )
])

# 2. Read raw JSON
raw_redemptions = spark.table(
    "skypoints_db.raw_redemption_json_documents"
)

# 3. Parse JSON and retain parsing status
parsed_redemptions = raw_redemptions.withColumn(
    "parsed",
    F.from_json(F.col("raw_json"), redemption_schema)
)

# 4. Separate malformed JSON documents
malformed_json = parsed_redemptions.filter(
    F.col("parsed").isNull()
)

# 5. Flatten valid JSON documents
exploded_df = (
    parsed_redemptions
    .filter(F.col("parsed").isNotNull())
    .select(
        F.col("parsed.member_id").alias("member_id"),
        F.col("parsed.feed_date").alias("raw_feed_date"),
        F.explode_outer("parsed.redemptions").alias("redemption"),
        "source_file_name",
        "load_timestamp"
    )
)

# 6. Extract and convert transaction fields
flattened_all = exploded_df.select(
    "member_id",
    F.to_date("raw_feed_date", "yyyyMMdd").alias("feed_date"),
    F.col("redemption.txn_id").alias("txn_id"),
    F.to_date(
        F.col("redemption.txn_date"), "yyyyMMdd"
    ).alias("txn_date"),
    F.col("redemption.partner").alias("partner"),
    F.col("redemption.miles_redeemed").alias("miles_redeemed"),
    F.col("redemption.status").alias("status"),
    "source_file_name",
    "load_timestamp"
)

# 7. Apply data-quality validations
validated_df = flattened_all.withColumn(
    "validation_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("member_id").isNull() |
            (F.trim(F.col("member_id")) == ""),
            F.lit("Missing member ID")
        ),
        F.when(
            F.col("feed_date").isNull(),
            F.lit("Invalid feed date")
        ),
        F.when(
            F.col("txn_id").isNull() |
            (F.trim(F.col("txn_id")) == ""),
            F.lit("Missing transaction ID")
        ),
        F.when(
            F.col("txn_date").isNull(),
            F.lit("Invalid transaction date")
        ),
        F.when(
            F.col("partner").isNull() |
            (F.trim(F.col("partner")) == ""),
            F.lit("Missing partner")
        ),
        F.when(
            F.col("miles_redeemed").isNull() |
            (F.col("miles_redeemed") < 0),
            F.lit("Invalid miles redeemed")
        ),
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("Missing transaction status")
        )
    )
)

# 8. Separate valid and rejected transactions
valid_transactions = validated_df.filter(
    F.col("validation_reason") == ""
).drop("validation_reason")

rejected_transactions = validated_df.filter(
    F.col("validation_reason") != ""
)

# 9. Add processing timestamp
flattened_df = valid_transactions.withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

# 10. Load valid transactions
flattened_df.createOrReplaceTempView("redemption_ready")

spark.sql("""
    INSERT OVERWRITE TABLE skypoints_db.redemption_transactions
    SELECT
        member_id,
        feed_date,
        txn_id,
        txn_date,
        partner,
        miles_redeemed,
        status,
        source_file_name,
        load_timestamp,
        processed_timestamp
    FROM redemption_ready
""")

# 11. Display results
print("Raw JSON documents:", raw_redemptions.count())
print("Malformed JSON documents:", malformed_json.count())
print("Valid transactions:", flattened_df.count())
print("Rejected transactions:", rejected_transactions.count())

print("REJECTED TRANSACTIONS")
display(rejected_transactions)

print("LOADED TRANSACTIONS")
display(
    spark.table("skypoints_db.redemption_transactions")
)


Raw JSON documents: 1
Malformed JSON documents: 0
Valid transactions: 2
Rejected transactions: 0
REJECTED TRANSACTIONS


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,validation_reason


LOADED TRANSACTIONS


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,processed_timestamp
223457,2024-01-15,RX10091,2024-01-10,AeroLink,12000,COMPLETED,skypoints_redemptions_sample.json,2026-11-02T13:36:29.832Z,2026-11-02T13:37:56.910Z
223457,2024-01-15,RX10092,2024-01-13,SkyPoints,5000,PENDING,skypoints_redemptions_sample.json,2026-11-02T13:36:29.832Z,2026-11-02T13:37:56.910Z


In [0]:

from pyspark.sql import functions as F

members = spark.table("skypoints_db.member_current")
redemptions = spark.table("skypoints_db.redemption_transactions")

# 1. Member record count
member_count = members.count()

# 2. Mandatory member field validation
invalid_members = members.filter(
    F.col("member_name").isNull() |
    (F.trim(F.col("member_name")) == "") |
    F.col("member_id").isNull() |
    (F.trim(F.col("member_id")) == "") |
    F.col("enrollment_date").isNull()
)

# 3. Duplicate member ID validation
duplicate_members = (
    members
    .groupBy("member_id")
    .count()
    .filter(
        F.col("member_id").isNotNull() &
        (F.col("count") > 1)
    )
)

# 4. Duplicate transaction ID validation
duplicate_transactions = (
    redemptions
    .groupBy("txn_id")
    .count()
    .filter(
        F.col("txn_id").isNotNull() &
        (F.col("count") > 1)
    )
)

# 5. Redemption data validation
invalid_redemptions = redemptions.filter(
    F.col("member_id").isNull() |
    (F.trim(F.col("member_id")) == "") |
    F.col("txn_id").isNull() |
    (F.trim(F.col("txn_id")) == "") |
    F.col("feed_date").isNull() |
    F.col("txn_date").isNull() |
    F.col("partner").isNull() |
    (F.trim(F.col("partner")) == "") |
    F.col("miles_redeemed").isNull() |
    (F.col("miles_redeemed") < 0) |
    F.col("status").isNull() |
    (F.trim(F.col("status")) == "")
)

# 6. Find redemptions without matching members
unmatched_redemptions = redemptions.join(
    members.select("member_id"),
    on="member_id",
    how="left_anti"
)

# 7. Country table reconciliation
country_table_names = [
    "target_usa",
    "target_ind",
    "target_phil",
    "target_can",
    "target_au"
]

country_total = sum(
    spark.table(f"skypoints_db.{table}").count()
    for table in country_table_names
)

country_reconciliation = (
    "PASSED" if country_total == member_count else "FAILED"
)

# 8. Collect validation results
results = [
    ("Total members", member_count),
    ("Invalid mandatory member records", invalid_members.count()),
    ("Duplicate member IDs", duplicate_members.count()),
    ("Total redemption transactions", redemptions.count()),
    ("Duplicate transaction IDs", duplicate_transactions.count()),
    ("Invalid redemption records", invalid_redemptions.count()),
    ("Unmatched redemption records", unmatched_redemptions.count()),
    ("Country table records", country_total),
]

print("DATA QUALITY VALIDATION SUMMARY")
print("--------------------------------")

for check, count in results:
    print(f"{check}: {count}")

print("Country reconciliation:", country_reconciliation)

# 9. Display exception records
print("\nDUPLICATE MEMBER RECORDS")
display(duplicate_members)

print("\nDUPLICATE TRANSACTIONS")
display(duplicate_transactions)

print("\nINVALID REDEMPTION RECORDS")
display(invalid_redemptions)

print("\nUNMATCHED REDEMPTIONS")
display(unmatched_redemptions)


DATA QUALITY VALIDATION SUMMARY
--------------------------------
Total members: 5
Invalid mandatory member records: 0
Duplicate member IDs: 0
Total redemption transactions: 2
Duplicate transaction IDs: 0
Invalid redemption records: 0
Unmatched redemption records: 0
Country table records: 5
Country reconciliation: PASSED

DUPLICATE MEMBER RECORDS


member_id,count



DUPLICATE TRANSACTIONS


txn_id,count



INVALID REDEMPTION RECORDS


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,processed_timestamp



UNMATCHED REDEMPTIONS


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,processed_timestamp


In [0]:

from pyspark.sql import functions as F

# 1. Read current member master
current_df = spark.table("skypoints_db.member_current")

# 2. Create a simulated country movement for Elena
test_update = (
    current_df
    .filter(F.col("member_id") == "223457")
    .withColumn("country", F.lit("IND"))
    .withColumn("state", F.lit("KA"))
    .withColumn(
        "load_timestamp",
        F.expr("load_timestamp + INTERVAL 1 DAY")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
)

# 3. Verify test record exists
if test_update.count() == 0:
    raise ValueError("Elena's member record was not found.")

# 4. Merge updated record into member master
test_update.createOrReplaceTempView("member_movement_test")

spark.sql("""
    MERGE INTO skypoints_db.member_current AS target
    USING member_movement_test AS source
    ON target.member_id = source.member_id

    WHEN MATCHED AND source.load_timestamp > target.load_timestamp
    THEN UPDATE SET *
""")

# 5. Synchronize country tables
country_tables = {
    "USA": "target_usa",
    "IND": "target_ind",
    "PHIL": "target_phil",
    "CAN": "target_can",
    "AU": "target_au"
}

updated_master = spark.table("skypoints_db.member_current")

for country, table_name in country_tables.items():

    country_df = updated_master.filter(
        F.upper(F.trim(F.col("country"))) == country
    )

    country_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"skypoints_db.{table_name}")

# 6. Verify Elena's updated master record
print("ELENA'S UPDATED MASTER RECORD")

elena = updated_master.filter(
    F.col("member_id") == "223457"
)

display(elena)

# 7. Verify country movement
usa_count = spark.table(
    "skypoints_db.target_usa"
).filter(F.col("member_id") == "223457").count()

ind_count = spark.table(
    "skypoints_db.target_ind"
).filter(F.col("member_id") == "223457").count()

print("Elena in USA table:", usa_count)
print("Elena in IND table:", ind_count)

# 8. Final movement validation
movement_status = (
    "PASSED" if usa_count == 0 and ind_count == 1
    else "FAILED"
)

print("Country movement validation:", movement_status)


ELENA'S UPDATED MASTER RECORD


member_name,member_id,enrollment_date,last_flight_date,tier_code,agent_name,state,country,dob,is_active,age,stale_member,source_file_name,load_timestamp,processed_timestamp
Elena,223457,2010-10-12,2012-10-13,GLD,Sam,KA,IND,1985-03-05,A,41,true,skypoints_members_sample.txt,2026-11-08T10:05:42.206Z,2026-11-02T13:38:13.837Z


Elena in USA table: 0
Elena in IND table: 1
Country movement validation: PASSED


In [0]:

from pyspark.sql import functions as F

# 1. Read current master
master_df = spark.table("skypoints_db.member_current")

# 2. Combine all country tables
country_tables = [
    "target_usa",
    "target_ind",
    "target_phil",
    "target_can",
    "target_au"
]

country_dfs = [
    spark.table(f"skypoints_db.{table_name}")
    for table_name in country_tables
]

country_df = country_dfs[0]

for table_df in country_dfs[1:]:
    country_df = country_df.unionByName(table_df)

# 3. Check duplicate members across country tables
duplicate_country_members = (
    country_df
    .groupBy("member_id")
    .count()
    .filter(F.col("count") > 1)
)

# 4. Check master members missing from country tables
missing_country_members = master_df.join(
    country_df.select("member_id"),
    on="member_id",
    how="left_anti"
)

# 5. Check country records missing from master
orphan_country_members = country_df.join(
    master_df.select("member_id"),
    on="member_id",
    how="left_anti"
)

# 6. Check country assignment consistency
incorrect_country_assignments = (
    country_df.alias("c")
    .join(
        master_df.select(
            "member_id",
            F.col("country").alias("master_country")
        ).alias("m"),
        on="member_id",
        how="inner"
    )
    .filter(
        ~F.col("c.country").eqNullSafe(F.col("m.master_country"))
    )
)

# 7. Read redemption transactions
redemptions_df = spark.table(
    "skypoints_db.redemption_transactions"
)

# 8. Join redemption transactions with member profiles
redemption_profile_df = redemptions_df.join(
    master_df.select(
        "member_id",
        "member_name",
        "country",
        "tier_code"
    ),
    on="member_id",
    how="left"
)

# 9. Check unmatched redemption transactions
unmatched_redemptions = redemption_profile_df.filter(
    F.col("member_name").isNull()
)

# 10. Overall redemption summary
redemption_summary = redemptions_df.agg(
    F.count("*").alias("total_transactions"),
    F.sum("miles_redeemed").alias("total_miles_redeemed"),
    F.countDistinct("member_id").alias("unique_members")
)

# 11. Redemption summary by country and tier
redemption_by_country_tier = (
    redemption_profile_df
    .groupBy("country", "tier_code")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("miles_redeemed").alias("total_miles_redeemed")
    )
    .orderBy("country", "tier_code")
)

# 12. Final reconciliation
master_count = master_df.count()
country_count = country_df.count()
duplicate_count = duplicate_country_members.count()
missing_count = missing_country_members.count()
orphan_count = orphan_country_members.count()
incorrect_count = incorrect_country_assignments.count()
redemption_count = redemptions_df.count()
profile_count = redemption_profile_df.count()
unmatched_count = unmatched_redemptions.count()

print("FINAL END-TO-END RECONCILIATION")
print("--------------------------------")
print("Master members:", master_count)
print("Country table records:", country_count)
print("Duplicate country members:", duplicate_count)
print("Missing country members:", missing_count)
print("Orphan country members:", orphan_count)
print("Incorrect country assignments:", incorrect_count)
print("Redemption transactions:", redemption_count)
print("Joined redemption records:", profile_count)
print("Unmatched redemptions:", unmatched_count)

print("\nCOUNTRY RECONCILIATION:",
      "PASSED" if (
          master_count == country_count
          and duplicate_count == 0
          and missing_count == 0
          and orphan_count == 0
          and incorrect_count == 0
      ) else "FAILED")

print("REDEMPTION JOIN VALIDATION:",
      "PASSED" if (
          redemption_count == profile_count
          and unmatched_count == 0
      ) else "FAILED")

print("\nOVERALL REDEMPTION SUMMARY")
display(redemption_summary)

print("\nREDEMPTION SUMMARY BY COUNTRY AND TIER")
display(redemption_by_country_tier)

print("\nREDEMPTION WITH MEMBER PROFILE")
display(redemption_profile_df)

print("\nDUPLICATE COUNTRY MEMBERS")
display(duplicate_country_members)

print("\nMISSING COUNTRY MEMBERS")
display(missing_country_members)

print("\nORPHAN COUNTRY MEMBERS")
display(orphan_country_members)

print("\nINCORRECT COUNTRY ASSIGNMENTS")
display(incorrect_country_assignments)

print("\nUNMATCHED REDEMPTIONS")
display(unmatched_redemptions)


FINAL END-TO-END RECONCILIATION
--------------------------------
Master members: 5
Country table records: 5
Duplicate country members: 0
Missing country members: 0
Orphan country members: 0
Incorrect country assignments: 0
Redemption transactions: 2
Joined redemption records: 2
Unmatched redemptions: 0

COUNTRY RECONCILIATION: PASSED
REDEMPTION JOIN VALIDATION: PASSED

OVERALL REDEMPTION SUMMARY


total_transactions,total_miles_redeemed,unique_members
2,17000,1



REDEMPTION SUMMARY BY COUNTRY AND TIER


country,tier_code,total_transactions,total_miles_redeemed
IND,GLD,2,17000



REDEMPTION WITH MEMBER PROFILE


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,processed_timestamp,member_name,country,tier_code
223457,2024-01-15,RX10091,2024-01-10,AeroLink,12000,COMPLETED,skypoints_redemptions_sample.json,2026-11-02T13:36:29.832Z,2026-11-02T13:37:56.910Z,Elena,IND,GLD
223457,2024-01-15,RX10092,2024-01-13,SkyPoints,5000,PENDING,skypoints_redemptions_sample.json,2026-11-02T13:36:29.832Z,2026-11-02T13:37:56.910Z,Elena,IND,GLD


... (6 outputs left out)


INCORRECT COUNTRY ASSIGNMENTS


member_id,member_name,enrollment_date,last_flight_date,tier_code,agent_name,state,country,dob,is_active,age,stale_member,source_file_name,load_timestamp,processed_timestamp,master_country



UNMATCHED REDEMPTIONS


member_id,feed_date,txn_id,txn_date,partner,miles_redeemed,status,source_file_name,load_timestamp,processed_timestamp,member_name,country,tier_code


# SkyPoints – Data Analyst Assessment

## 1. Project Overview

The objective of this project is to process, validate, transform and analyze SkyPoints membership and redemption data using Python, PySpark, SQL and Databricks.

The analysis focuses on maintaining accurate member information, identifying data-quality issues, tracking country changes, processing redemption transactions and validating the consistency of the final datasets.

## 2. Data Processing and Analysis Approach

### Member Data Analysis

1. Loaded the source member flat file into raw storage.
2. Parsed the source records into structured columns using PySpark.
3. Converted date fields into appropriate date formats.
4. Calculated member age using Date of Birth.
5. Identified stale members based on the last flight date.
6. Validated mandatory fields, date values and field lengths.
7. Separated valid and invalid records for further analysis.
8. Maintained the latest available member information in the master dataset.
9. Analyzed and synchronized member records across country-specific tables.

### Redemption Data Analysis

1. Loaded the redemption JSON source data.
2. Parsed the JSON structure using an explicit schema.
3. Flattened the nested redemption array into individual transaction records.
4. Analyzed transaction IDs, dates, partners, redeemed miles and transaction statuses.
5. Joined redemption transactions with member information using `member_id`.
6. Used the combined dataset to analyze redemption activity along with current member details.

## 3. Data Quality and Validation

Implemented data-quality checks to ensure the reliability and consistency of the datasets.

The checks include:

- Mandatory field validation.
- Duplicate member ID identification.
- Invalid date detection.
- Field-length validation.
- Duplicate redemption transaction identification.
- Invalid redemption record identification.
- Unmatched member reference identification.
- Country-wise record reconciliation.
- Missing and duplicate member identification across country tables.

Invalid member records are separated into a quarantine table with rejection reasons for investigation.

The supplied sample dataset passed the implemented validation checks.

## 4. Business Rules and Analytical Assumptions

- Member source dates use `yyyyMMdd` format, except DOB, which uses `MMddyyyy`.
- Member age is calculated using Date of Birth.
- A member is classified as stale when the last flight date is more than 90 days before the processing date.
- The latest available member record determines the member's current country.
- Country-specific datasets are synchronized using the current member master.
- Redemption transactions are associated with member profiles using `member_id`.
- Original source values are retained rather than silently modifying questionable identifiers or country codes.

## 5. Source Data Limitations and Assumptions

**Post Code:** The data dictionary includes Post_Code, but the supplied sample flat-file layout does not contain this field. It has therefore not been fabricated or populated.

**Member key:** The data dictionary identifies Member_Name as a key and Member_ID as non-key. However, the redemption JSON provides member_id, and this implementation uses Member_ID as the operational identifier for matching and deduplication. This definition should be confirmed with the source data owner.

**Member identifiers:** Some sample